#### Prepared for Gabor's Data Analysis

### Data Analysis for Business, Economics, and Policy
by Gabor Bekes and  Gabor Kezdi
 
Cambridge University Press 2021

**[gabors-data-analysis.com ](https://gabors-data-analysis.com/)**

 License: Free to share, modify and use for educational purposes. 
 Not to be used for commercial purposes.


### CHAPTER 20
**CH20A Working from home and employee performance**

using the wfh dataset

version 1.0 2021-05-05

In [1]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.core.display import HTML
from plotnine import *
from stargazer.stargazer import Stargazer

warnings.filterwarnings("ignore")


In [2]:
# Current script folder
current_path = os.getcwd()
dirname = current_path.split("da_case_studies")[0]

# location folders
data_in = dirname + "da_data_repo/working-from-home/clean/"
data_out = dirname + "da_case_studies/ch20-working-from-home/"
output = dirname + "da_case_studies/ch20-working-from-home/output/"

func = dirname + "da_case_studies/ch00-tech-prep/"
sys.path.append(func)


### Load data

In [135]:
data = pd.read_csv("/workspaces/codespaces-jupyter/data/wfh_tidy_person.csv")


In [136]:
data.columns

Index(['personid', 'treatment', 'ordertaker', 'type', 'quitjob', 'age',
       'costofcommute', 'children', 'male', 'married', 'perform10',
       'perform11', 'prior_experience', 'tenure', 'basewage', 'bonus',
       'grosswage', 'ageyoungestchild', 'rental', 'bedroom',
       'second_technical', 'high_school', 'tertiary_technical', 'university',
       'internet', 'phonecalls0', 'phonecalls1'],
      dtype='object')

In [137]:
data.columns

Index(['personid', 'treatment', 'ordertaker', 'type', 'quitjob', 'age',
       'costofcommute', 'children', 'male', 'married', 'perform10',
       'perform11', 'prior_experience', 'tenure', 'basewage', 'bonus',
       'grosswage', 'ageyoungestchild', 'rental', 'bedroom',
       'second_technical', 'high_school', 'tertiary_technical', 'university',
       'internet', 'phonecalls0', 'phonecalls1'],
      dtype='object')

In [138]:
data["phonecalls1"]

0       0.000
1      21.797
2       0.000
3       0.000
4       0.000
        ...  
244     0.539
245    12.975
246     0.000
247     9.589
248    10.459
Name: phonecalls1, Length: 249, dtype: float64

### Balance

In [139]:
data["ageyoungestchild"] = np.where(
    data["children"] == 0, np.nan, data["ageyoungestchild"]
)


In [140]:
data["ageyoungestchild"]

0      10.0
1       NaN
2      11.0
3       NaN
4       0.0
       ... 
244     NaN
245     NaN
246     NaN
247     NaN
248     NaN
Name: ageyoungestchild, Length: 249, dtype: float64

In [141]:
# convertin nonnumeric dummies to numeric
data["ordertaker"] = data["ordertaker"].astype(int)
#data["children"] = np.where(data["children"] == "yes", 1, 0)
#data["rental"] = np.where(data["rental"] == "yes", 1, 0)
#data["bedroom"] = np.where(data["bedroom"] == "yes", 1, 0)
#data["ageyoungestchild"] = pd.to_numeric(data["ageyoungestchild"])


In [142]:
data.columns

Index(['personid', 'treatment', 'ordertaker', 'type', 'quitjob', 'age',
       'costofcommute', 'children', 'male', 'married', 'perform10',
       'perform11', 'prior_experience', 'tenure', 'basewage', 'bonus',
       'grosswage', 'ageyoungestchild', 'rental', 'bedroom',
       'second_technical', 'high_school', 'tertiary_technical', 'university',
       'internet', 'phonecalls0', 'phonecalls1'],
      dtype='object')

In [143]:
variables = data.columns.tolist()

In [144]:
variables

['personid',
 'treatment',
 'ordertaker',
 'type',
 'quitjob',
 'age',
 'costofcommute',
 'children',
 'male',
 'married',
 'perform10',
 'perform11',
 'prior_experience',
 'tenure',
 'basewage',
 'bonus',
 'grosswage',
 'ageyoungestchild',
 'rental',
 'bedroom',
 'second_technical',
 'high_school',
 'tertiary_technical',
 'university',
 'internet',
 'phonecalls0',
 'phonecalls1']

In [145]:
data.head().T

,0,1,2,3,4
personid,3906.000000,4122.000000,4448.000000,4942.000000,5018.000000
treatment,1.000000,1.000000,0.000000,1.000000,1.000000
ordertaker,0.000000,1.000000,0.000000,0.000000,0.000000
type,2.000000,1.000000,2.000000,4.000000,4.000000
quitjob,0.000000,0.000000,0.000000,0.000000,0.000000
age,33.000000,30.000000,35.000000,27.000000,29.000000
costofcommute,8.000000,18.000000,10.000000,20.000000,15.000000
children,1.000000,0.000000,1.000000,0.000000,1.000000
male,0.000000,0.000000,0.000000,0.000000,0.000000
married,1.000000,0.000000,1.000000,1.000000,1.000000


In [146]:
data.dtypes

personid                int64
treatment               int64
ordertaker              int64
type                    int64
quitjob                 int64
age                     int64
costofcommute         float64
children                int64
male                    int64
married                 int64
perform10             float64
perform11             float64
prior_experience      float64
tenure                float64
basewage              float64
bonus                 float64
grosswage             float64
ageyoungestchild      float64
rental                  int64
bedroom                 int64
second_technical        int64
high_school             int64
tertiary_technical      int64
university              int64
internet                int64
phonecalls0           float64
phonecalls1           float64
dtype: object

In [147]:
mean_t = dict()
mean_c = dict()
sd = dict()
p_value = dict()


In [148]:
for i in variables:
    # Regression model
    print(f"{i}: type={type(data[i])}, shape={data[i].shape}")
    data[i] = pd.to_numeric(data[i], errors='coerce')
    model = smf.ols(formula="{i}~treatment".format(i=i), data=data).fit()

    # Mean control
    mean_c[i] = data.loc[data["treatment"] == 0, i].dropna().mean()
    # Mean treated
    mean_t[i] = data.loc[data["treatment"] == 1, i].dropna().mean()
    # p-value from regression
    p_value[i] = model.pvalues[1]
    # Standard deviation
    sd[i] = data[i].dropna().std()


personid: type=<class 'pandas.core.series.Series'>, shape=(249,)
treatment: type=<class 'pandas.core.series.Series'>, shape=(249,)
ordertaker: type=<class 'pandas.core.series.Series'>, shape=(249,)
type: type=<class 'pandas.core.series.Series'>, shape=(249,)
quitjob: type=<class 'pandas.core.series.Series'>, shape=(249,)
age: type=<class 'pandas.core.series.Series'>, shape=(249,)
costofcommute: type=<class 'pandas.core.series.Series'>, shape=(249,)
children: type=<class 'pandas.core.series.Series'>, shape=(249,)
male: type=<class 'pandas.core.series.Series'>, shape=(249,)
married: type=<class 'pandas.core.series.Series'>, shape=(249,)
perform10: type=<class 'pandas.core.series.Series'>, shape=(249,)
perform11: type=<class 'pandas.core.series.Series'>, shape=(249,)
prior_experience: type=<class 'pandas.core.series.Series'>, shape=(249,)
tenure: type=<class 'pandas.core.series.Series'>, shape=(249,)
basewage: type=<class 'pandas.core.series.Series'>, shape=(249,)
bonus: type=<class 'pand

In [149]:
table = pd.DataFrame([mean_t, mean_c, sd, p_value]).T.round(2)
table.columns = [
    "Treatment mean",
    "Control mean",
    "Std.dev.",
    "p-value of test of equal means",
]

In [150]:
table


,Treatment mean,Control mean,Std.dev.,p-value of test of equal means
personid,30299.21,28865.02,11844.94,0.34
treatment,1.00,0.00,0.50,0.00
ordertaker,0.52,0.56,0.50,0.53
type,2.05,1.91,1.27,0.37
quitjob,0.16,0.35,0.43,0.00
age,24.44,24.35,3.55,0.85
costofcommute,7.89,8.34,6.96,0.61
children,0.11,0.24,0.38,0.01
male,0.47,0.47,0.50,0.99
married,0.22,0.32,0.44,0.07


 ### outcomes

In [152]:
# quit firm during 8 months of experiment
# phone calls worked, for order takers


In [153]:
quitjobs = (
    data.groupby("treatment")
    .agg(mean=("quitjob", "mean"), std=("quitjob", "std"), N=("quitjob", "count"))
    .round(3)
)


In [154]:
total_quitjob = data.agg(
    mean_total=("quitjob", "mean"),
    std_total=("quitjob", "std"),
    N_total=("quitjob", "count"),
).T.round(3)


In [155]:
quitjobs


,mean,std,N
treatment,,,
0,0.347,0.478,118
1,0.160,0.368,131


In [156]:
total_quitjob


,mean_total,std_total,N_total
quitjob,0.249,0.433,249.0


In [159]:
# Looking at the data columns, 'perform10' appears to be the metric for phone calls
# for order takers mentioned in the comments in cell 26
phonecalls1 = (
    data.query("ordertaker==1")
    .groupby("treatment")
    .agg(
        mean=("phonecalls1", "mean"),
        std=("phonecalls1", "std"),
        N=("phonecalls1", "count"),
    )
    .round(2)
)


In [164]:
total_phonecalls = (
    data.query("ordertaker==1")
    .agg(
        mean_total=("phonecalls1", "mean"),
        std_total=("phonecalls1", "std"),
        N_total=("phonecalls1", "count"),
    )
    .T.round(2)
)


In [161]:
phonecalls1


,mean,std,N
treatment,,,
0,10.06,6.10,66
1,14.10,5.31,68


In [165]:
total_phonecalls


,mean_total,std_total,N_total
phonecalls1,12.11,6.04,134.0


In [166]:
# Bar chart for quit rates


In [167]:
data["quit_pct"] = data["quitjob"] * 100
data["stayed_pct"] = (1 - data["quitjob"]) * 100


In [168]:
barchart_data = pd.melt(
    data[["treatment", "quit_pct", "stayed_pct"]]
    .groupby("treatment")
    .agg({"quit_pct": "mean", "stayed_pct": "mean"})
    .reset_index(),
    id_vars="treatment",
).rename(columns={"variable": "employees", "value": "pct"})


In [169]:
barchart_data["treatment"] = np.where(
    barchart_data["treatment"] == 0, "Non-treatment group", "Treatment group"
)


In [ ]:
quitrates_barchart = (
    ggplot(barchart_data, aes(fill="employees", y="pct", x="treatment"))
    + geom_bar(stat="identity")
    + theme_bw()
    + labs(y="Share of employees (percent)", x="")
    + scale_x_discrete()
    + scale_fill_manual(
        labels=("Quit", "Stayed"), name=" ", values=(color[1], color[0])
    )
)

quitrates_barchart


In [38]:
quitrates_barchart = (
    ggplot(barchart_data, aes(fill="employees", y="pct", x="treatment"))
    + geom_bar(stat="identity")
    + theme_bw()
    + labs(y="Share of employees (percent)", x="")
    + scale_x_discrete()
    + scale_fill_manual(
        labels=("Quit", "Stayed"), name=" ", values=(color[1], color[0])
    )
)

quitrates_barchart


NameError: name 'color' is not defined

### Regression analysis 
 Outcome variables: 1) quit firm during 8 months of experiment , 2) phone calls worked, for ordertakers

In [ ]:
# Outcomes by treatment

# 1) Quit firm
quitjobs


,mean,std,N
treatment,,,
0,0.347,0.478,118
1,0.160,0.368,131


In [ ]:
# 2) Phonecalls (ordertakers only)
phonecalls1


,mean,std,N
treatment,,,
0,10.06,6.10,66
1,14.10,5.31,68


### Regression 1: ATE estimates, no covariates

In [ ]:
reg1 = smf.ols(formula="quitjob~treatment", data=data).fit(cov_type="HC1")
reg2 = smf.ols(formula="phonecalls1~treatment", data=data.query("ordertaker==1")).fit(
    cov_type="HC1"
)


In [ ]:
stargazer = Stargazer([reg1, reg2])
stargazer.rename_covariates({"Intercept": "Constant"})
HTML(stargazer.render_html())


### Regression 2: ATE estimates, with covariates of some unbalance

In [ ]:
reg3 = smf.ols(
    formula="quitjob ~ treatment + married + children + internet", data=data
).fit(cov_type="HC1")
reg4 = smf.ols(
    formula="phonecalls1 ~ treatment + married + children + internet",
    data=data.query("ordertaker==1"),
).fit(cov_type="HC1")


In [ ]:
stargazer = Stargazer([reg3, reg4])
stargazer.rename_covariates({"Intercept": "Constant"})
HTML(stargazer.render_html())
